# *Global Settings*

In [ ]:
import pandas as pd
import numpy as np
import osmnx as ox
import geopandas as gpd
from pathlib import Path
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 0. CONFIGURACIÓN DE RUTAS PORTABLES
# ==============================================================================
BASE_DIR = Path.cwd()

# --- DEFINICIÓN DE CARPETAS --- 
INPUTS_DIR = BASE_DIR / "Inputs"
INFRA_DIR = INPUTS_DIR / "Infrastructure"
RATES_DIR = INPUTS_DIR / "Emission rates"
GPS_DIR = INPUTS_DIR / "GPS User Data"

OUTPUTS_DIR = BASE_DIR / "Outputs"
INTERMEDIATE_DIR = OUTPUTS_DIR / "Intermediate Outputs"
FINAL_DIR = OUTPUTS_DIR / "Final Outputs"

# Crear carpetas automáticamente
for folder in [INFRA_DIR, RATES_DIR, GPS_DIR, INTERMEDIATE_DIR, FINAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# --- DEFINICIÓN DE ARCHIVOS ESPECÍFICOS ---
FILE_GRAFO = INFRA_DIR / "monterrey_drive_network_V1.pkl"
FILE_GRAFO_WALK = INFRA_DIR / "monterrey_walk_network_EydanV1.pkl" 

FILE_METRO = INFRA_DIR / "lineas_metrorrey.csv"
FILE_BUS = INFRA_DIR / "rutas_transporte.geojson"
FILE_GPS_ORIGINAL = GPS_DIR / "top_20users_1_month.parquet"
FILE_MOVES = RATES_DIR / "cleaned_emission_rates_formatted_SB.parquet"

# Archivos de Salida
FILE_INTERMEDIO = INTERMEDIATE_DIR / "Rutas_Completadas_Clasificadas.parquet"
FILE_FINAL_PARQUET = FINAL_DIR / "Resultados_Datos_Emisiones_GPS.parquet"
FILE_FINAL_CSV = FINAL_DIR / "Resultados_Mapa_Emisiones_GPS_Kepler.csv"
FILE_FINAL_GEOJSON = FINAL_DIR / "Resultados_Mapa_Emisiones_GPS_Kepler.geojson"

print(f"Carpetas listas en: {BASE_DIR}")

# **MODULE 1: Código de Modos de Transporte**

## M1 Input files and settings

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import KDTree
from shapely import wkt
import shapely.geometry

print("Cargando infraestructura de transporte...")
bus_routes = gpd.read_file(FILE_BUS)
subway_df = pd.read_csv(FILE_METRO)

# Convertimos la columna de texto a geometría espacial
if 'geometry' in subway_df.columns:
    subway_df['geometry'] = subway_df['geometry'].apply(wkt.loads)
    subway_routes = gpd.GeoDataFrame(subway_df, geometry='geometry', crs="EPSG:4326")
elif 'WKT' in subway_df.columns:
    subway_df['geometry'] = subway_df['WKT'].apply(wkt.loads)
    subway_routes = gpd.GeoDataFrame(subway_df, geometry='geometry', crs="EPSG:4326")
elif 'lat' in subway_df.columns and 'lon' in subway_df.columns:
    subway_routes = gpd.GeoDataFrame(subway_df, geometry=gpd.points_from_xy(subway_df.lon, subway_df.lat), crs="EPSG:4326")
else:
    raise ValueError("¡ALERTA! Revisa las columnas del archivo del metro.")

# ---------------------------------------------------------
# CORRECCIÓN METODOLÓGICA: PROYECCIÓN Y CACHÉ ESPACIAL
# ---------------------------------------------------------
bus_routes = bus_routes.to_crs("EPSG:32614")
subway_routes = subway_routes.to_crs("EPSG:32614")

print("Generando Índices Espaciales (R-Trees) globales para alto rendimiento...")
# Al llamar .sindex, GeoPandas pre-calcula el árbol espacial de geometrías complejas (líneas)
# Queda guardado en la memoria del objeto, evitando que se recalcule por cada usuario.
_ = subway_routes.sindex 
_ = bus_routes.sindex
print("Infraestructura lista, proyectada y optimizada espacialmente.")

print("Infraestructura lista y proyectada a metros reales.")

## M1 Functions y Asignación Modal

In [ ]:
def calcular_cercania_infraestructura(df, subway_routes, bus_routes):
    """Calcula cercanía al metro/bus usando distancias MÉTRICAS exactas a la línea."""
    
    # 1. Proyectar temporalmente a UTM 14N
    gdf_pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
    gdf_pts = gdf_pts.to_crs("EPSG:32614")
    
    RADIO_BUSQUEDA_METROS = 100 
    
    # 2. Búsqueda Espacial Exacta (sjoin_nearest mide a la LÍNEA, no al centroide)
    # --- METRO ---
    metro_join = gpd.sjoin_nearest(gdf_pts, subway_routes, how='left', distance_col='dist_metro')
    metro_join = metro_join[~metro_join.index.duplicated(keep='first')] # Limpiar empates de distancia
    df['near_subway_line'] = (metro_join['dist_metro'] < RADIO_BUSQUEDA_METROS).astype(int)
    
    # --- BUS ---
    bus_join = gpd.sjoin_nearest(gdf_pts, bus_routes, how='left', distance_col='dist_bus')
    bus_join = bus_join[~bus_join.index.duplicated(keep='first')] 
    df['near_bus_route'] = (bus_join['dist_bus'] < RADIO_BUSQUEDA_METROS).astype(int)
    
    return df

def clasificar_viajes(df):
    """Modelo Bayesiano completo (4 variables) original, 100% Vectorizado y Normalizado."""
    modos = ['Carro', 'Bus', 'Metro', 'Caminar']
    
    # --- MATRICES ORIGINALES (Tomadas de 4_Clasificación.ipynb) ---
    Cercania = np.array([[0.10, 0.10, 0.80, 0.00],
                        [0.10, 0.80, 0.00, 0.10],
                        [0.40, 0.25, 0.05, 0.30]])

    Velocidad = np.array([[0.05, 0.10, 0.15, 0.60],
                        [0.47, 0.38, 0.05, 0.10],
                        [0.50, 0.30, 0.20, 0.00],
                        [1.00, 0.00, 0.00, 0.00]])

    Distancia = np.array([[0.10, 0.20, 0.30, 0.40],
                        [0.25, 0.25, 0.30, 0.20],
                        [0.40, 0.15, 0.25, 0.20],
                        [0.60, 0.30, 0.00, 0.10],
                        [0.40, 0.40, 0.00, 0.20]])

    Velprom = np.array([[0.10, 0.10, 0.20, 0.60],
                        [0.40, 0.25, 0.25, 0.10]])

    # --- PREPARAR VARIABLES FALTANTES POR VIAJE ---
    # Calculamos la distancia total y velocidad promedio del viaje al vuelo para este usuario
    mask_viajes = df['trip'] > 0
    df['dist_total_km'] = 0.0
    df['avg_speed_trip'] = 0.0
    
    if mask_viajes.any():
        df.loc[mask_viajes, 'dist_total_km'] = df[mask_viajes].groupby('trip')['dis lineal [m]'].transform('sum') / 1000.0
        df.loc[mask_viajes, 'avg_speed_trip'] = df[mask_viajes].groupby('trip')['Speed [km/h]'].transform('mean')

    # --- PASO 1: ÍNDICES VECTORIZADOS (Filtros de variables) ---
    # Cercanía: 0 (Metro), 1 (Bus), 2 (Ninguno)
    idx_c = np.where(df['near_subway_line'] == 1, 0, 
            np.where(df['near_bus_route'] == 1, 1, 2))

    idx_v = np.digitize(df['Speed [km/h]'].fillna(0), bins=[6.001, 20.001, 80.001])
    idx_d = np.digitize(df['dist_total_km'], bins=[1.0, 6.001, 10.001, 18.001])
    idx_vp = np.digitize(df['avg_speed_trip'], bins=[6.001])
    
    # --- PASO 2: BAYES Y NORMALIZACIÓN POR PUNTOS ---
    # Multiplicación matricial (No normalizada)
    P_unnorm = Cercania[idx_c] * Velocidad[idx_v] * Distancia[idx_d] * Velprom[idx_vp]
    
    # Normalización: Hacemos que las probabilidades de los 4 modos sumen 1.0 en cada fila
    suma_puntos = P_unnorm.sum(axis=1, keepdims=True)
    suma_puntos[suma_puntos == 0] = 1 # Prevenir división por cero
    P_norm_puntos = P_unnorm / suma_puntos
    
    df_probs = pd.DataFrame(P_norm_puntos, columns=modos, index=df.index)
    df_probs['trip'] = df['trip']
    
    # --- PASO 3: SUMA DE PROBABILIDADES Y VOTACIÓN POR VIAJE ---
    viajes_df = df_probs[df_probs['trip'] > 0]
    
    if not viajes_df.empty:
        # Acumulamos el peso probabilístico de todos los puntos GPS dentro de un mismo viaje
        suma_por_viaje = viajes_df.groupby('trip')[modos].sum()
        # Seleccionamos el modo que obtuvo la mayor probabilidad global para ese viaje
        modos_ganadores = suma_por_viaje.idxmax(axis=1) 
        df['modo_transporte'] = df['trip'].map(modos_ganadores)
    else:
        df['modo_transporte'] = np.nan
        
    # --- PASO 4: LIMPIEZA DE PARADAS Y MEMORIA ---
    df.loc[df['trip'] <= 0, 'modo_transporte'] = 'Parada'
    df.drop(columns=['dist_total_km', 'avg_speed_trip'], inplace=True, errors='ignore')
    
    return df

# **MODULE 2: Completado de Rutas**

Version 7.

### M2 Input files and settings

In [ ]:
import pandas as pd
import numpy as np
import osmnx as ox
import igraph as ig
import json
import geopandas as gpd
from joblib import Parallel, delayed
from datetime import timedelta
import pyproj
from shapely.ops import transform, substring
from shapely import wkt
from shapely.geometry import Point, LineString
from pyproj import Transformer

# ---------------------------------------------------------
# CORRECCIÓN DE RENDIMIENTO: TRANSFORMADORES GLOBALES
# Se instancian globalmente UNA SOLA VEZ, ahorrando miles de operaciones.
# ---------------------------------------------------------
TRANSFORMER_TO_UTM = Transformer.from_crs("EPSG:4326", "EPSG:32614", always_xy=True)
TRANSFORMER_TO_WGS = Transformer.from_crs("EPSG:32614", "EPSG:4326", always_xy=True)

# Guardamos una versión proyectada global del metro para la interpolación
subway_routes_proj = subway_routes.to_crs("EPSG:32614")

### M2 Functions

In [ ]:
def get_ids(df):
    return np.array(df['caid'].unique())

def assign_trips(df):
    """
    Segmentación con Filtro Anti-Teletransportación y 
    Acumulador de Tiempo Estacionario (Low-Pass Filter) Corregido.
    """
    df = df.copy()
    df['Speed [km/h]'] = df['Speed [km/h]'].fillna(0.0)
    df['dis lineal [m]'] = df['dis lineal [m]'].fillna(0.0)
    df['travel time'] = df['travel time'].fillna(pd.Timedelta(seconds=0))

    speeds = df['Speed [km/h]'].tolist()
    travel_times = df['travel time'].dt.total_seconds().tolist() 
    distances = df['dis lineal [m]'].tolist()

    trips = []
    trip_counter = 0
    stop_counter = 0
    
    STOP_SPEED = 3.0  
    STOP_TIME = 300   
    T_MAX_TELEPORT = 1800 

    accumulated_stop_time = 0

    for i in range(len(speeds)):
        speed = speeds[i]
        tt = travel_times[i]
        dist = distances[i]
        
        previous_trip = trips[-1] if i > 0 else None

        # 1. Filtro Anti-Teletransportación (Saltos gigantes de GPS)
        if tt > T_MAX_TELEPORT:
            stop_counter -= 1
            current_trip = stop_counter
            accumulated_stop_time = 0
            
        else:
            # 2. Detección de quietud
            if speed < STOP_SPEED and dist < 100:
                accumulated_stop_time += tt # Sumamos el tiempo de este pequeño ping
                
                if accumulated_stop_time > STOP_TIME:
                    # Ya estuvo quieto más de 5 mins acumulados. Es una Parada real.
                    if previous_trip is None or previous_trip > 0:
                        stop_counter -= 1
                        current_trip = stop_counter
                    else:
                        current_trip = previous_trip # Se mantiene en la misma parada
                else:
                    # Está quieto, pero lleva menos de 5 mins (ej. Semáforo rojo). Sigue siendo Viaje.
                    current_trip = previous_trip if previous_trip is not None else 1
                    
            # 3. Detección de movimiento
            else:
                accumulated_stop_time = 0 # Arrancó el auto, el contador de quietud vuelve a cero
                
                if previous_trip is None or previous_trip < 0:
                    trip_counter += 1
                    current_trip = trip_counter
                else:
                    current_trip = previous_trip # Sigue en el mismo viaje
                    
        trips.append(current_trip)
        
    df['trip'] = trips
    return df


# Cálculo de distancia geodésica mediante fórmula de Haversine vectorizada
def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    distance = 6371 * c  
    return distance

# Filtrado de identificadores con densidad de datos insuficiente
def delete_ids_with_few_rows(df, id_list, threshold=2):
    id_counts = df[df['caid'].isin(id_list)]['caid'].value_counts()
    valid_ids = id_counts[id_counts > threshold].index
    return df[df['caid'].isin(valid_ids)]


def _crear_curva_metro(p_start, p_end, df_metro_proj, num_points):
    """Interpola un trayecto de metro obligándolo a seguir la línea específica correcta."""
    
    # 1. Proyectar a UTM 14N usando el transformer global
    x_start, y_start = TRANSFORMER_TO_UTM.transform(p_start.x, p_start.y)
    x_end, y_end = TRANSFORMER_TO_UTM.transform(p_end.x, p_end.y)
    pt_start = Point(x_start, y_start)
    pt_end = Point(x_end, y_end)
    
    # 2. Identificar a qué línea específica de metro pertenecen
    distancias_start = df_metro_proj.distance(pt_start)
    linea_mas_cercana_idx = distancias_start.idxmin()
    nearest_line = df_metro_proj.loc[linea_mas_cercana_idx, 'geometry']
    
    # 3. Proyectar distancias SOLO sobre esa línea particular
    d_start = nearest_line.project(pt_start)
    d_end = nearest_line.project(pt_end)
    
    # 4. Extraer el segmento exacto (soportando viajes en ambas direcciones)
    if d_start <= d_end:
        sub_line = substring(nearest_line, d_start, d_end)
    else:
        sub_line = substring(nearest_line, d_end, d_start)
        # Invertir coordenadas para que la lista siga la dirección del viaje real
        sub_line = LineString(list(sub_line.coords)[::-1])
    
    # 5. Generar puntos interpolados equidistantes
    length = sub_line.length
    if length <= 0 or num_points <= 1:
        pts_utm = [pt_start for _ in range(num_points)]
    else:
        distancias = np.linspace(0, length, num_points)
        pts_utm = [sub_line.interpolate(d) for d in distancias]
        
    # 6. Devolver a WGS84 (Lat/Lon)
    pts_wgs = []
    for pt in pts_utm:
        lon, lat = TRANSFORMER_TO_WGS.transform(pt.x, pt.y)
        pts_wgs.append(Point(lon, lat))
        
    return pts_wgs

def complete_route(id, registros_person, 
                   G_drive, G_ig_drive, dict_nx_to_ig_drive, dict_ig_to_nx_drive,
                   G_walk, G_ig_walk, dict_nx_to_ig_walk, dict_ig_to_nx_walk,
                   geometry_metro=None):
    """
    Realiza el map-matching segmento por segmento.
    Decide dinámicamente si usar el grafo de autos o el de peatones.
    """
    
    rpc_list = [] 
    
    exitos_ruteo = 0
    fallos_ruteo = 0

    trips_arr = registros_person['trip'].to_numpy()
    lats_arr = registros_person['latitude'].to_numpy()
    lons_arr = registros_person['longitude'].to_numpy()
    speeds_arr = registros_person['Speed [km/h]'].to_numpy()
    nodes_drive_arr = registros_person['node_drive'].to_numpy()
    nodes_walk_arr = registros_person['node_walk'].to_numpy()
    dis_arr = registros_person['dis lineal [m]'].to_numpy()
    
    timestamps_list = registros_person['local_timestamp'].tolist() 
    
    if 'modo_transporte' in registros_person.columns:
        modos_arr = registros_person['modo_transporte'].to_numpy()
    else:
        modos_arr = np.array(['Carro'] * len(registros_person))

    for row in range(len(registros_person) - 1):
        trip_id = trips_arr[row] 
        modo_actual = modos_arr[row]
        dis_lineal_precalculada = dis_arr[row]
        
        do_routing = False
        es_error_ruteo = False 
        
        # LÓGICA DE SELECCIÓN DE GRAFO
        es_peaton = (str(modo_actual).lower() == 'caminar')
        
        if es_peaton:
            G_actual = G_walk
            G_ig_actual = G_ig_walk
            dict_nx_to_ig_actual = dict_nx_to_ig_walk
            dict_ig_to_nx_actual = dict_ig_to_nx_walk
            start_node = nodes_walk_arr[row]
            end_node = nodes_walk_arr[row+1]
        else:
            G_actual = G_drive
            G_ig_actual = G_ig_drive
            dict_nx_to_ig_actual = dict_nx_to_ig_drive
            dict_ig_to_nx_actual = dict_ig_to_nx_drive
            start_node = nodes_drive_arr[row]
            end_node = nodes_drive_arr[row+1]

        if trip_id > 0:
            if modo_actual == 'Metro':
                do_routing = False 
            elif start_node != end_node:
                do_routing = True

        if do_routing:
            current_time = timestamps_list[row]
            try:
                ig_start = dict_nx_to_ig_actual[start_node]
                ig_end = dict_nx_to_ig_actual[end_node]
                res = G_ig_actual.get_shortest_paths(ig_start, to=ig_end, weights='weight', output='epath')
                path_ig_edges = res[0] if res else []
                
                if len(path_ig_edges) > 0:
                    route = [start_node]
                    keys_path = []
                    for e_idx in path_ig_edges:
                        edge = G_ig_actual.es[e_idx]
                        keys_path.append(edge['nx_key'])
                        route.append(dict_ig_to_nx_actual[edge.target])
                else:
                    route = None
            except KeyError:
                route = None

            if (route is None) or (len(route) < 2):
                do_routing = False 
                es_error_ruteo = True 
                fallos_ruteo += 1
            else:
                try:
                    route_time, speed_route, osmids, geometrias, longitudes_calle_m, highways = [], [], [], [], [], []
                    coordinates = {'latitude': [], 'longitude': []}
                    
                    for (u, v), k in zip(zip(route[:-1], route[1:]), keys_path):
                        edge_dict = G_actual.get_edge_data(u, v)
                        edge_data = edge_dict[k]
                        
                        route_time.append(edge_data.get("travel_time", 0))
                        speed_route.append(edge_data.get("speed_kph", 0))
                        osmids.append(str(edge_data.get("osmid", "N/A")))
                        longitudes_calle_m.append(edge_data.get("length", 0))
                        
                        hw = edge_data.get("highway", "footway" if es_peaton else "unclassified")
                        if isinstance(hw, list): hw = hw[0]
                        highways.append(str(hw))
                        
                        if 'geometry' in edge_data:
                            geometrias.append(edge_data['geometry'].wkt)
                        else:
                            lon_u, lat_u = G_actual.nodes[u]['x'], G_actual.nodes[u]['y']
                            lon_v, lat_v = G_actual.nodes[v]['x'], G_actual.nodes[v]['y']
                            geometrias.append(f'LINESTRING ({lon_u} {lat_u}, {lon_v} {lat_v})')
                            
                        coordinates['latitude'].append(G_actual.nodes[u]['y'])
                        coordinates['longitude'].append(G_actual.nodes[u]['x'])

                    time_calc = sum(route_time)
                    time_real = (timestamps_list[row+1] - timestamps_list[row]).total_seconds()
                    exitos_ruteo += 1 
                    
                except Exception:
                    do_routing = False
                    es_error_ruteo = True
                    fallos_ruteo += 1

            if do_routing:
                for idx, (lat, lon, t_row, s_row, node_id, osmid, geom, l_row, hw_row) in enumerate(zip(
                    coordinates['latitude'], coordinates['longitude'], route_time, 
                    speed_route, route[:-1], osmids, geometrias, longitudes_calle_m, highways)):
                    
                    if time_calc > 0:
                        time_alloc = time_real * (t_row / time_calc)
                    else:
                        time_alloc = time_real / len(route_time) if len(route_time) > 0 else 0
                        
                    if time_alloc > 0:
                        speed_kph = (l_row / 1000.0) / (time_alloc / 3600.0)
                    else:
                        speed_kph = 0
                        
                    MAX_SPEED_KPH = 150.0 
                    if speed_kph > MAX_SPEED_KPH:
                        speed_kph = MAX_SPEED_KPH
                        time_alloc = (l_row / 1000.0) / MAX_SPEED_KPH * 3600.0 

                    rpc_list.append({
                        'caid': id,
                        'trip': trip_id,    
                        'latitude': lat,
                        'longitude': lon,
                        'Speed [km/h]': speed_kph,
                        'local_timestamp': current_time,
                        'start_node': node_id,
                        'end_node': route[idx+1],
                        'osmid': osmid,
                        'highway': hw_row, 
                        'geometry': geom,
                        'distance_m': l_row,
                        'modo_transporte': modo_actual
                    })
                    
                    t_delta = pd.Timedelta(seconds=time_alloc)
                    current_time = current_time + t_delta

        if not do_routing:
            lat1, lon1 = lats_arr[row], lons_arr[row]
            lat2, lon2 = lats_arr[row+1], lons_arr[row+1]
            speed = speeds_arr[row]
            current_time = timestamps_list[row]
            node = start_node 
            osmid = 'N/A'
            
            if trip_id > 0 and dis_lineal_precalculada > 0:
                if modo_actual == 'Metro' and geometry_metro is not None:
                    try:
                        pt_start = Point(lon1, lat1)
                        pt_end = Point(lon2, lat2)
                        pts_wgs = _crear_curva_metro(pt_start, pt_end, geometry_metro, num_points=10)
                        geom = LineString(pts_wgs).wkt
                        osmid = 'metro_line'
                    except Exception as e:
                        geom = f'LINESTRING ({lon1} {lat1}, {lon2} {lat2})'
                else:
                    geom = f'LINESTRING ({lon1} {lat1}, {lon2} {lat2})'
            else:
                geom = f'POINT ({lon1} {lat1})'
                
            hw_final = 'unclassified'
            if es_error_ruteo:
                hw_final = 'routing_error'
            
            rpc_list.append({
                'caid': id, 
                'trip': trip_id,         
                'latitude': lat1, 
                'longitude': lon1, 
                'Speed [km/h]': speed, 
                'local_timestamp': current_time,
                'start_node': node, 
                'end_node': node, 
                'osmid': osmid,
                'highway': hw_final, 
                'geometry': geom,
                'distance_m': dis_lineal_precalculada,
                'modo_transporte': modo_actual 
            })

    return pd.DataFrame(rpc_list)

## 2. Pipeline de Ejecución

#### 2.1. GPS Data

In [ ]:
print("Cargando datos GPS de usuarios...")
df = pd.read_parquet(FILE_GPS_ORIGINAL)

# ==============================================================================
# CORRECCIÓN DE ZONA HORARIA (UTC -> MONTERREY)
# ==============================================================================
print("Ajustando reloj satelital a hora local de Monterrey...")

# 1. Detectar el nombre de la columna de tiempo 
col_tiempo = 'utc_timestamp' if 'utc_timestamp' in df.columns else 'date'

# 2. Asegurar formato datetime (¡CORRECCIÓN CLAVE: Agregamos unit='s'!)
try:
    df[col_tiempo] = pd.to_datetime(df[col_tiempo], unit='s')
except ValueError:
    df[col_tiempo] = pd.to_datetime(df[col_tiempo])

# 3. Aplicar el desplazamiento horario inteligente 
if df[col_tiempo].dt.tz is None:
    df[col_tiempo] = df[col_tiempo].dt.tz_localize('UTC').dt.tz_convert('America/Monterrey')
else:
    df[col_tiempo] = df[col_tiempo].dt.tz_convert('America/Monterrey')

# 4. Quitar la etiqueta 'tz' para Parquet
df[col_tiempo] = df[col_tiempo].dt.tz_localize(None)

# 5. Renombrar
if 'utc_timestamp' in df.columns:
    df = df.rename(columns={'utc_timestamp': 'local_timestamp'})
    col_tiempo = 'local_timestamp'
elif 'date' in df.columns:
    df = df.rename(columns={'date': 'local_timestamp'})
    col_tiempo = 'local_timestamp'

print(f"Reloj ajustado. Columna de tiempo actualizada a: '{col_tiempo}'")
# ==============================================================================

# Si se quiere limitar a x usuarios (NO BORRAR):
num_users = 20
usuarios_unicos = df['caid'].unique()[:num_users]
df = df[df['caid'].isin(usuarios_unicos)].copy()

print('Total records before filtering:', len(df))

# Delimitación espacial (Geofencing): Área metropolitana de Monterrey
ref_lat, ref_lon = 25.6866, -100.3161
threshold_distance = 48

# Cálculo de proximidad y filtrado por radio de influencia
df['Distance'] = haversine_vectorized(ref_lat, ref_lon, df['latitude'].values, df['longitude'].values)
df = df[df['Distance'] < threshold_distance]

df = delete_ids_with_few_rows(df, get_ids(df), threshold=2)

print('Total valid records after filtering:', len(df))
print('Total valid IDs:', len(get_ids(df)))

# ==============================================================================
# SEGMENTACIÓN Y LIMPIEZA TEMPORAL (INCLUYE PROTECCIÓN DE DÍAS)
# ==============================================================================

# Eliminación de redundancias temporales
df = df.drop_duplicates(subset=['caid', 'local_timestamp'])

# Segmentación temporal y ordenamiento
df['date'] = df['local_timestamp'].dt.date
df = df.sort_values(by=['caid', 'local_timestamp'])

# --- IMPLEMENTACIÓN OPCIÓN 1 BLINDADA: ELIMINAR DÍAS INCOMPLETOS ---
dias_unicos = df['date'].unique()
fecha_min = df['date'].min()
fecha_max = df['date'].max()

print(f"Rango original detectado: {fecha_min} al {fecha_max} ({len(dias_unicos)} días distintos en total).")

# EL ESCUDO: Solo recortamos si tenemos al menos 3 días distintos
if len(dias_unicos) >= 3:
    print("Aplicando limpieza de bordes para conservar solo días 100% completos...")
    df = df[(df['date'] > fecha_min) & (df['date'] < fecha_max)]
    print(f"Rango útil para análisis: {df['date'].min()} al {df['date'].max()}")
else:
    print("PROTECCIÓN ACTIVADA: El archivo tiene menos de 3 días.")
    print("No se recortarán los bordes para evitar vaciar la base de datos.")
    print("Nota para el análisis: Estos días podrían estar incompletos en sus extremos horarios.")

# ==============================================================================
# CÁLCULO DE VECTORES DE DESPLAZAMIENTO Y VELOCIDAD
# ==============================================================================

# Generación de vectores de desplazamiento 
df['lat_end'] = df.groupby(['caid', 'date'])['latitude'].shift(-1)
df['lon_end'] = df.groupby(['caid', 'date'])['longitude'].shift(-1)
df['local_timestamp_end'] = df.groupby(['caid', 'date'])['local_timestamp'].shift(-1)

# Exclusión de registros sin pareja
df = df.dropna(subset=['lat_end', 'lon_end', 'local_timestamp_end'])

# Cálculo de distancia geodésica
df['dis lineal [m]'] = haversine_vectorized(
    df['latitude'].values, df['longitude'].values, 
    df['lat_end'].values, df['lon_end'].values
) * 1000

# Determinación de tiempo de viaje
df['travel time'] = df['local_timestamp_end'] - df['local_timestamp']

# Estimación de velocidad escalar instantánea
df['speed [m/s]'] = np.where(
    df['travel time'].dt.total_seconds() > 0, 
    df['dis lineal [m]'] / df['travel time'].dt.total_seconds(), 
    0
)

df['Speed [km/h]'] = df['speed [m/s]'] * 3.6

# Filtrado por umbral de velocidad física
max_speed_threshold = 150
df = df[df['Speed [km/h]'] <= max_speed_threshold]

df = df.drop(['lat_end', 'lon_end', 'local_timestamp_end', 'speed [m/s]', 'Distance'], axis=1)
df = df.reset_index(drop=True)

df.head()

### 2.2. Road Network Data

In [ ]:
import os
import osmnx as ox
import pickle
import igraph as ig

# --- Modifica la celda donde cargas el grafo ---
if FILE_GRAFO.exists():
    print("Cargando grafo de AUTOS...")
    with open(FILE_GRAFO, 'rb') as f:
        G_drive = pickle.load(f)
        
    # USAR GRAFO DE AUTOS COMO PLACEHOLDER TEMPORAL PARA WALK
    # Cuando termine tu descarga, cambia Path(r"...") por la ruta de tu nuevo .pkl
    FILE_GRAFO_WALK = Path(r"D:\EV_Completado de rutas\NB Output_EydanV1\monterrey_walk_network_EydanV1.pkl")
    
    if FILE_GRAFO_WALK.exists():
        print("Cargando grafo de PEATONES real...")
        with open(FILE_GRAFO_WALK, 'rb') as f:
            G_walk = pickle.load(f)
    else:
        print("IMPORTANT: Grafo de peatones no encontrado. Usando el de autos como temporal.")
        G_walk = G_drive
        
def convert_osmnx_to_igraph(nx_graph):
    print("Iniciando traducción a igraph...")
    nodes = list(nx_graph.nodes())
    nx_to_ig = {node_id: i for i, node_id in enumerate(nodes)}
    ig_to_nx = {i: node_id for i, node_id in enumerate(nodes)}
    
    g_ig = ig.Graph(directed=True)
    g_ig.add_vertices(len(nodes))
    
    edges, weights, nx_keys = [], [], []
    for u, v, key, data in nx_graph.edges(keys=True, data=True):
        edges.append((nx_to_ig[u], nx_to_ig[v]))
        weights.append(data.get('travel_time', 1.0)) 
        nx_keys.append(key)
        
    g_ig.add_edges(edges)
    g_ig.es['weight'] = weights
    g_ig.es['nx_key'] = nx_keys
    
    print("Grafo vectorizado exitosamente.")
    return g_ig, nx_to_ig, ig_to_nx

# LA CONEXIÓN CRÍTICA: Ejecutamos la función para AMBOS mapas
print("--- Procesando Grafo Vehicular ---")
G_ig_drive, dict_nx_to_ig_drive, dict_ig_to_nx_drive = convert_osmnx_to_igraph(G_drive)

print("--- Procesando Grafo Peatonal ---")
G_ig_walk, dict_nx_to_ig_walk, dict_ig_to_nx_walk = convert_osmnx_to_igraph(G_walk)      

### 2.3. Completado de Rutas

In [ ]:
%%time
from joblib import Parallel, delayed
import pandas as pd
import geopandas as gpd
import osmnx as ox

print("Proyectando grafos y calculando índices espaciales globales...")
# 1. Preparación espacial ultra-rápida para AMBOS mapas
# --- Grafo Vehicular ---
G_drive_proj = ox.project_graph(G_drive)
# --- Grafo Peatonal ---
G_walk_proj = ox.project_graph(G_walk)

# Proyectar los puntos del usuario
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
# Proyectamos a la misma CRS que los grafos (asumimos que ambos tienen la misma zona UTM)
gdf_proj = gdf.to_crs(G_drive_proj.graph['crs'])

print("Asignando nodos más cercanos (Vehiculares y Peatonales)...")
# El punto se mapea al grafo global de calles
df['node_drive'] = ox.distance.nearest_nodes(G_drive_proj, gdf_proj.geometry.x.values, gdf_proj.geometry.y.values)
# El punto se mapea al grafo global de banquetas/parques
df['node_walk'] = ox.distance.nearest_nodes(G_walk_proj, gdf_proj.geometry.x.values, gdf_proj.geometry.y.values)

# Liberar memoria pesada antes del paralelo
del G_drive_proj, G_walk_proj, gdf, gdf_proj

print("Indexando el DataFrame...")
df_idx = df.set_index(['caid', 'date']).sort_index()
tareas_keys = df_idx.index.unique().tolist()


# 2. Función envoltorio para cada hilo del procesador
def process_day_wrapper(id_usuario, fecha):
    try:
        # Extraer los datos específicos de este usuario y día
        df_dia = df_idx.loc[(id_usuario, fecha)].copy()
    except KeyError:
        return pd.DataFrame()

    if isinstance(df_dia, pd.Series):
        df_dia = df_dia.to_frame().T

    df_dia = df_dia.reset_index()

    if df_dia.empty:
        return pd.DataFrame() 

    try:
        # --- PASO 1: SEGMENTACIÓN ---
        # Separamos los pings en viajes y paradas
        df_dia = assign_trips(df_dia)
        
        # --- PASO 2: CLASIFICACIÓN MODAL ---
        if 'trip' in df_dia.columns:  
            # Usamos las variables globales de infraestructura (subway_routes, bus_routes)
            df_dia = calcular_cercania_infraestructura(df_dia, subway_routes, bus_routes)
            df_dia = clasificar_viajes(df_dia)
        else:
            df_dia['modo_transporte'] = 'Carro' # Fallback real si falla la segmentación
            
        # --- PASO 3: COMPLETADO DE RUTAS ---
        # Pasamos los dos grafos y diccionarios a la nueva función
        df_routed = complete_route(
            id_usuario, df_dia, 
            G_drive, G_ig_drive, dict_nx_to_ig_drive, dict_ig_to_nx_drive,
            G_walk, G_ig_walk, dict_nx_to_ig_walk, dict_ig_to_nx_walk,
            geometry_metro=subway_routes_proj
        )
        return df_routed

    except Exception as e:
        print(f"Error procesando {id_usuario} el {fecha}: {e}")
        return pd.DataFrame()


print(f"Iniciando procesamiento paralelo de {len(tareas_keys)} tareas (Combinación Usuario/Día)...")

# 3. EJECUCIÓN PARALELA
results = Parallel(n_jobs=5, backend="loky", verbose=30, batch_size=1)(
    delayed(process_day_wrapper)(caid, fecha) for caid, fecha in tareas_keys
)

# 4. CONSOLIDACIÓN DE RESULTADOS
print("Agregando resultados y consolidando topología...")
results_clean = [r for r in results if not r.empty]
df_completed_routes = pd.concat(results_clean, axis=0, ignore_index=True)

if 'date' in df_completed_routes.columns:
    df_completed_routes = df_completed_routes.drop(columns=['date'])

# Asegurar que los tipos de datos sean correctos
df_completed_routes['caid'] = df_completed_routes['caid'].astype(str)
df_completed_routes['trip'] = df_completed_routes['trip'].astype(int)

df_completed_routes = df_completed_routes.reset_index(drop=True)
print("Completado de rutas finalizado con éxito.")

#### Module 2 Sanity Check

In [ ]:
# ==============================================================================
# SANITY CHECK: MÓDULOS 1 Y 2 (MODOS Y RUTEO)
# ==============================================================================
print("="*60)
print("REPORTE DE DIAGNÓSTICO: RUTAS Y MODOS")
print("="*60)

total_registros = len(df_completed_routes)
print(f"▶ Total de puntos generados: {total_registros:,}")

print("\n▶ 1A. Distribución Modal (Por cantidad de registros):")
modos_pct_regs = df_completed_routes['modo_transporte'].value_counts(normalize=True) * 100
print(modos_pct_regs.round(2).astype(str) + " %")

print("\n▶ 1B. Distribución Modal (Por distancia viajada):")
distancia_por_modo = df_completed_routes.groupby('modo_transporte')['distance_m'].sum()
distancia_pct = (distancia_por_modo / distancia_por_modo.sum()) * 100
print(distancia_pct.sort_values(ascending=False).round(2).astype(str) + " %")

print("\n▶ 2. Distribución de vías (Top 5 tipos):")
vias_pct = df_completed_routes['highway'].value_counts(normalize=True) * 100
print(vias_pct.head(5).round(2).astype(str) + " %")

# Alerta de fallos silenciosos
if 'routing_error' in df_completed_routes['highway'].values:
    errores = (df_completed_routes['highway'] == 'routing_error').sum()
    pct_errores = (errores / total_registros) * 100
    print(f"\n▶ ▶ ADVERTENCIA: {errores:,} segmentos ({pct_errores:.2f}%) fallaron el map-matching")
else:
    print("\nÉXITO: 0 segmentos fallaron el map-matching.")
print("="*60)

### 2.4. Guardado de Completado de Rutas

In [ ]:
print("Guardando rutas completadas para el Módulo 3...")
# Usamos la variable FILE_INTERMEDIO
df_completed_routes.to_parquet(FILE_INTERMEDIO)
print(f"Archivo intermedio guardado en: {FILE_INTERMEDIO.name}")

# **MODULE 3: Código de Cálculo de Emisiones**

## Settings y Cálculo Dinámico 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
print("Iniciando Módulo 3: Cálculo de emisiones.")

COLUMNA_MODO = 'modo_transporte' 
COLUMNA_VELOCIDAD = 'Speed [km/h]'
COLUMNA_DISTANCIA = 'distance_m'

print("Cargando rutas y tasas de emisión...")

# Cargamos el archivo puente del Módulo 2
df_rutas = pd.read_parquet(FILE_INTERMEDIO)

# Cargamos la tabla de MOVES
df_emisiones_raw = pd.read_parquet(FILE_MOVES)

# INCLUSIÓN DE TODOS LOS CONTAMINANTES CRITERIO
POLLUTANTS = ['CO', 'CO2', 'CO2_Equiv', 'HC', 'NOx', 'PM10', 'PM25']

OSM_TO_MOVES_ROAD = {
    'motorway': 4, 'motorway_link': 4, 'trunk': 4, 'trunk_link': 4, 
    'primary': 4, 'primary_link': 4, 'secondary': 5, 'secondary_link': 5, 
    'tertiary': 5, 'tertiary_link': 5, 'unclassified': 5, 'residential': 5,
    'routing_error': 5 
}

# ==============================================================================
# 2. PREPARACIÓN DE VARIABLES
# ==============================================================================
print("Preparando variables temporales y espaciales...")
df_rutas['orden_original'] = range(len(df_rutas))

df_rutas['local_timestamp'] = pd.to_datetime(df_rutas['local_timestamp'])

# Extracción de Mes, Hora y Día
df_rutas['Month'] = df_rutas['local_timestamp'].dt.month
df_rutas['Hour'] = df_rutas['local_timestamp'].dt.hour + 1
df_rutas['Day'] = np.where(df_rutas['local_timestamp'].dt.dayofweek < 5, 5, 2)
    
df_rutas['Road'] = df_rutas['highway'].map(OSM_TO_MOVES_ROAD).fillna(5).astype(int)

# Clasificación de Source ID
condiciones = [
    df_rutas['trip'] < 0,
    df_rutas[COLUMNA_MODO].astype(str).str.contains('(?i)carro|auto', na=False),
    df_rutas[COLUMNA_MODO].astype(str).str.contains('(?i)bus', na=False)
]
opciones = [0, 21, 42]
df_rutas['Source'] = np.select(condiciones, opciones, default=0)

# SpeedBins
df_rutas['avg_speed_mph'] = df_rutas[COLUMNA_VELOCIDAD] * 0.621371
bins = [0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, float('inf')]
df_rutas['SpeedBin'] = pd.cut(df_rutas['avg_speed_mph'], bins=bins, labels=list(range(1, 17)), right=False)
df_rutas['SpeedBin'] = df_rutas['SpeedBin'].astype(object).fillna(1).astype(int)

# ==============================================================================
# 3. CRUCE CON MOVES EMISSION RATES
# ==============================================================================
print("Cruzando con factores de emisión...")

merge_cols = ['Day', 'Hour', 'Road', 'Source', 'SpeedBin']
cols_needed = merge_cols + POLLUTANTS
df_emisiones_raw = df_emisiones_raw[cols_needed].astype({
    'Day': int, 'Hour': int, 'Road': int, 'Source': int, 'SpeedBin': int
})

# Optimización de la tabla de emisiones
df_emisiones = df_emisiones_raw.groupby(merge_cols, as_index=False)[POLLUTANTS].mean()

df_motorizados = df_rutas[df_rutas['Source'] > 0].copy()
df_no_motorizados = df_rutas[df_rutas['Source'] == 0].copy()

# NIVEL 1: Cruce Exacto
df_motorizados = pd.merge(df_motorizados, df_emisiones, on=merge_cols, how='left')

mask_faltantes = df_motorizados[POLLUTANTS[0]].isna()
if mask_faltantes.any():
    print(f"▶ Imputando {mask_faltantes.sum():,} registros sin cruce exacto...")
    
    promedios_sb = df_emisiones.groupby(['Source', 'SpeedBin'])[POLLUTANTS].mean().reset_index()
    promedios_source = df_emisiones.groupby('Source')[POLLUTANTS].mean().reset_index()
    
    # NIVEL 2: Imputación por SpeedBin (Vecino cercano)
    fuentes_unicas = df_emisiones['Source'].unique()
    idx_completo = pd.MultiIndex.from_product([fuentes_unicas, range(1, 17)], names=['Source', 'SpeedBin'])
    df_curvas = promedios_sb.set_index(['Source', 'SpeedBin']).reindex(idx_completo).groupby(level='Source').ffill().groupby(level='Source').bfill().reset_index()
    
    df_curvas = df_curvas.rename(columns={p: f"{p}_curve" for p in POLLUTANTS})
    df_motorizados = pd.merge(df_motorizados, df_curvas, on=['Source', 'SpeedBin'], how='left')
    
    df_global = promedios_source.rename(columns={p: f"{p}_global" for p in POLLUTANTS})
    df_motorizados = pd.merge(df_motorizados, df_global, on='Source', how='left')
    
    # Aplicar cascada completa
    for p in POLLUTANTS:
        df_motorizados[p] = df_motorizados[p].fillna(df_motorizados[f"{p}_curve"])
        df_motorizados[p] = df_motorizados[p].fillna(df_motorizados[f"{p}_global"])
        df_motorizados[p] = df_motorizados[p].fillna(0.0) # Fallback final extremo
        
    df_motorizados = df_motorizados.drop(columns=[f"{p}_curve" for p in POLLUTANTS] + [f"{p}_global" for p in POLLUTANTS])

# Unir y limpiar
for p in POLLUTANTS: df_no_motorizados[p] = 0.0 
df_final = pd.concat([df_motorizados, df_no_motorizados], ignore_index=True).sort_values('orden_original').reset_index(drop=True)

# ==============================================================================
# 4. CÁLCULO DE TOTALES
# ==============================================================================
print("Calculando masa total de emisiones...")
df_final['distance_km_calc'] = df_final['distance_m'] / 1000.0

for p in POLLUTANTS:
    df_final[f"Densidad_{p}_g_km"] = df_final[p] 
    df_final[f"Total_{p}_g"] = df_final[f"Densidad_{p}_g_km"] * df_final['distance_km_calc']

df_final['fecha_kepler'] = df_final['local_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

print("Módulo 3 completado.")

#### Module 3 Sanity Check

In [ ]:
# ==============================================================================
# SANITY CHECK: MÓDULO 3 (EMISIONES MOVES)
# ==============================================================================
print("="*60)
print("REPORTE DE DIAGNÓSTICO: INVENTARIO DE EMISIONES")
print("="*60)

print("▶ 1. Mapeo de Identificadores (Source ID):")
resumen_source = df_final.groupby('Source').agg(
    Registros=('Source', 'size'),
    Distancia_km=('distance_km_calc', 'sum')
)
total_regs = resumen_source['Registros'].sum()
total_dist = resumen_source['Distancia_km'].sum()

for source, row in resumen_source.iterrows():
    pct_reg = (row['Registros'] / total_regs) * 100
    pct_dist = (row['Distancia_km'] / total_dist) * 100 if total_dist > 0 else 0
    nombre = 'No Motorizado/Parada' if source == 0 else ('Carro' if source == 21 else 'Bus')
    print(f"   - {nombre} (ID {source}): {row['Registros']:,} registros ({pct_reg:.2f}%) | {row['Distancia_km']:,.2f} km ({pct_dist:.2f}%)")

POLLUTANTS_noeqiv = ['CO', 'CO2', 'HC', 'NOx', 'PM10', 'PM25']

print("\n▶ 2. Inventario Total de Contaminantes (Global):")
totales_emisiones = {p: df_final[f"Total_{p}_g"].sum() for p in POLLUTANTS_noeqiv} 
serie_totales = pd.Series(totales_emisiones).sort_values(ascending=False)
total_masa = serie_totales.sum()

for p, masa in serie_totales.items():
    pct = (masa / total_masa) * 100 if total_masa > 0 else 0
    print(f"   - {p}: {masa:,.2f} g ({pct:.4f}%)")

print("\n▶ 3. Inventario Local de Calidad del Aire (EXCLUYENDO CO2 y CO2_Equiv):")
pol_excluidos = [p for p in POLLUTANTS if p not in ['CO2', 'CO2_Equiv']]
totales_sin_co2 = {p: df_final[f"Total_{p}_g"].sum() for p in pol_excluidos}
serie_sin_co2 = pd.Series(totales_sin_co2).sort_values(ascending=False)
total_masa_sin_co2 = serie_sin_co2.sum()

for p, masa in serie_sin_co2.items():
    pct = (masa / total_masa_sin_co2) * 100 if total_masa_sin_co2 > 0 else 0
    print(f"   - {p}: {masa:,.2f} g ({pct:.2f}%)")

print("\n▶ 4. Calidad del Cruce con MOVES (Efectividad del Modelo):")
# Filtramos solo los que usan motor (Carro, Bus)
motorizados = df_final[df_final['Source'] > 0]
total_motorizados = len(motorizados)

ceros_injustificados = motorizados[motorizados['Densidad_CO2_Equiv_g_km'] == 0]
segmentos_cero_dist = motorizados[motorizados['distance_m'] == 0]
fallos = len(ceros_injustificados)

if total_motorizados > 0:
    tasa_exito = ((total_motorizados - fallos) / total_motorizados) * 100
    print(f"   - Registros motorizados procesados: {total_motorizados:,}")
    print(f"   - Registros con factor de emisión asignado: {total_motorizados - fallos:,}")
    print(f"   - TASA DE ÉXITO DE MAPEO: {tasa_exito:.2f} %")
    
    if fallos > 0:
        print(f"   - Registros huérfanos (sin match en MOVES): {fallos:,} ({(fallos/total_motorizados)*100:.2f}%)")
else:
    print("   - No hay registros motorizados para evaluar.")

print(f"\n   Nota: Hay {len(segmentos_cero_dist):,} segmentos motorizados donde la distancia avanzada fue 0 metros.")
print("="*60)

## Guardado de Resultados Finales

#### In Parquet format

In [ ]:
df_final.to_parquet(FILE_FINAL_PARQUET)
print(f"Proceso terminado. Archivo de emisiones guardado en: {FILE_FINAL_PARQUET.name}")

#### In GeoJSON Format

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

print("Exportando mapa a GeoJSON para Kepler...")

# 1. Cargar el Parquet Maestro
df_mapa = pd.read_parquet(FILE_FINAL_PARQUET)

print("Seleccionando columnas para Kepler...")

# 2. Seleccionar SOLO las columnas que importan para visualizar 
columnas_mantener = ['caid', 'trip', 'fecha_kepler', 'Speed [km/h]', 'osmid', 'highway',
       'geometry', 'distance_m', 'modo_transporte', 'Densidad_CO_g_km', 'Total_CO_g', 'Densidad_CO2_g_km', 'Total_CO2_g',
       'Densidad_CO2_Equiv_g_km', 'Total_CO2_Equiv_g', 'Densidad_HC_g_km',
       'Total_HC_g', 'Densidad_NOx_g_km', 'Total_NOx_g', 'Densidad_PM10_g_km',
       'Total_PM10_g', 'Densidad_PM25_g_km', 'Total_PM25_g', 'latitude', 'longitude', 'start_node', 'end_node']

columnas_presentes = [c for c in columnas_mantener if c in df_mapa.columns]
df_mapa = df_mapa[columnas_presentes].copy()

# Remover filas donde no haya geometría
df_mapa = df_mapa.dropna(subset=['geometry'])

# ==============================================================================
# Convertir identificadores gigantes a texto para que el exportador C no colapse
columnas_gigantes = ['osmid', 'start_node', 'end_node', 'caid']
for col in columnas_gigantes:
    if col in df_mapa.columns:
        df_mapa[col] = df_mapa[col].astype(str)
# ==============================================================================

print(f"Construyendo el GeoJSON para {len(df_mapa)} segmentos (Optimizado en C)...")

# 3. Convertir los textos WKT a geometrías matemáticas reales y crear GeoDataFrame
geometrias = df_mapa['geometry'].apply(wkt.loads)
gdf_export = gpd.GeoDataFrame(df_mapa, geometry=geometrias, crs="EPSG:4326")

# 4. Exportar directo usando la variable global de la carpeta Outputs
print("Escribiendo archivo en disco duro...")
gdf_export.to_file(FILE_FINAL_GEOJSON, driver='GeoJSON')

# (Nota: corregí la variable del print final para que coincida con la que usaste al guardar)
print(f"Archivo GeoJSON guardado exitosamente en:\n'{FILE_FINAL_GEOJSON}'")

####    In CSV Format

In [ ]:
import pandas as pd

print("Exportando mapa a CSV para Kepler...")

df_mapa = pd.read_parquet(FILE_FINAL_PARQUET)

print("Seleccionando columnas para Kepler...")

columnas_mantener = ['caid', 'trip', 'fecha_kepler', 'Speed [km/h]', 'osmid', 'highway',
       'geometry', 'distance_m', 'modo_transporte', 'Densidad_CO_g_km', 'Total_CO_g', 'Densidad_CO2_g_km', 'Total_CO2_g',
       'Densidad_CO2_Equiv_g_km', 'Total_CO2_Equiv_g', 'Densidad_HC_g_km',
       'Total_HC_g', 'Densidad_NOx_g_km', 'Total_NOx_g', 'Densidad_PM10_g_km',
       'Total_PM10_g', 'Densidad_PM25_g_km', 'Total_PM25_g', 'latitude', 'longitude']

columnas_presentes = [c for c in columnas_mantener if c in df_mapa.columns]
df_mapa = df_mapa[columnas_presentes].copy()
df_mapa = df_mapa.dropna(subset=['geometry'])

print(f"Escribiendo {len(df_mapa)} segmentos a CSV...")
df_mapa.to_csv(FILE_FINAL_CSV, index=False) 

print(f"Archivo CSV guardado exitosamente en:\n'{FILE_FINAL_CSV}'")

### Results Visualization Dashboard

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ==============================================================================
# 5. DASHBOARD CIENTÍFICO: ANÁLISIS VISUAL DE RESULTADOS
# ==============================================================================
print("Generando Dashboard de Análisis Visual...")

df_final = pd.read_parquet(FILE_FINAL_PARQUET)

# Configuración de estilo visual (Estilo académico, limpio y profesional)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Análisis de Resulados - GPS Data Emissions', fontsize=20, weight='bold', y=1.02)

# --- GRÁFICA 1: Movilidad (Tiempo vs Distancia) ---
# Preparamos los datos
resumen_modo = df_final.groupby('Source').agg(
    Registros=('Source', 'size'),
    Distancia_km=('distance_km_calc', 'sum')
)
# Normalizamos a porcentajes
resumen_modo['% Tiempo (Registros)'] = (resumen_modo['Registros'] / resumen_modo['Registros'].sum()) * 100
resumen_modo['% Distancia Real'] = (resumen_modo['Distancia_km'] / resumen_modo['Distancia_km'].sum()) * 100
# Mapeo de nombres
nombres_source = {0: 'No Motorizado / Parada', 21: 'Carro', 42: 'Bus'}
resumen_modo.index = resumen_modo.index.map(nombres_source)

# Graficamos
resumen_modo[['% Tiempo (Registros)', '% Distancia Real']].plot(kind='bar', ax=axes[0, 0], color=['#3498db', '#2ecc71'])
axes[0, 0].set_title('Uso de Modos de Transporte', fontsize=14, weight='bold')
axes[0, 0].set_ylabel('Porcentaje (%)')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=0)

# --- GRÁFICA 2: Inventario de Tóxicos Locales (Sin CO2) ---
pol_locales = ['CO', 'NOx', 'HC', 'PM10', 'PM25']
masas_locales = [df_final[f"Total_{p}_g"].sum() / 1000 for p in pol_locales] # Convertimos a Kilogramos para la gráfica

sns.barplot(x=pol_locales, y=masas_locales, ax=axes[0, 1], palette="magma")
axes[0, 1].set_title('Inventario de Contaminantes Locales (Sin CO2)', fontsize=14, weight='bold')
axes[0, 1].set_ylabel('Masa Total Emitida (Kilogramos)')
axes[0, 1].set_xlabel('Contaminante')

# --- GRÁFICA 3: Dinámica Temporal (El pulso de la ciudad) ---
# Sumamos el CO2 por hora del día
emisiones_hora = df_final[df_final['Source'] > 0].groupby('Hour')['Total_CO2_Equiv_g'].sum() / 1000 # En kg
horas_completas = range(1, 25)
emisiones_hora = emisiones_hora.reindex(horas_completas, fill_value=0)

sns.lineplot(x=emisiones_hora.index, y=emisiones_hora.values, ax=axes[1, 0], color='#e74c3c', marker='o', linewidth=2.5)
axes[1, 0].set_title('Perfil Diurno de Emisiones', fontsize=14, weight='bold')
axes[1, 0].set_ylabel('CO2 Equivalente Emitido (kg)')
axes[1, 0].set_xlabel('Hora del Día (1-24)')
axes[1, 0].set_xticks(range(1, 25, 2)) # Marcas cada 2 horas

# --- GRÁFICA 4: Validación Científica (Curva de Emisión vs Velocidad) ---
# Filtramos solo los motorizados y calculamos la densidad promedio por SpeedBin
df_motorizado = df_final[df_final['Source'] > 0]
curva_velocidad = df_motorizado.groupby('SpeedBin')['Densidad_CO2_Equiv_g_km'].mean()

sns.lineplot(x=curva_velocidad.index, y=curva_velocidad.values, ax=axes[1, 1], color='#9b59b6', marker='s', linewidth=2.5)
axes[1, 1].set_title('Emisión vs. Velocidad', fontsize=14, weight='bold')
axes[1, 1].set_ylabel('Densidad Promedio (g CO2 / km)')
axes[1, 1].set_xlabel('SpeedBin (De menor a mayor velocidad)')
axes[1, 1].set_xticks(range(1, 17))

# Ajustamos el layout para que no se encimen los textos
# Ajustamos el layout para que no se encimen los textos
plt.tight_layout()

# ==============================================================================
# OPCIÓN PARA GUARDAR LA IMAGEN EN DISCO DURO
# ==============================================================================
# archivo_grafica = FINAL_DIR / "Dashboard_Emisiones_Resultados.png"
# plt.savefig(archivo_grafica, dpi=300, bbox_inches='tight')
# print(f"Dashboard guardado en alta calidad en: {archivo_grafica}")

# Mostramos las gráficas en el notebook
plt.show()

### Reduced CSVs

Light

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Las columnas exactas para visualizar en Kepler
columnas_mantener = [
    'caid', 'fecha_kepler', 'Speed [km/h]',
    'geometry', 'distance_m', 'modo_transporte', 'Total_CO2_g'
]

print("Cargando el archivo Parquet original (leyendo solo las columnas indicadas)...")

# ALERTA DE OPTIMIZACIÓN: 'columns' hace que el disco duro solo lea esas columnas específicas, ahorrando RAM
df_ligero = pd.read_parquet(FILE_FINAL_PARQUET, columns=columnas_mantener)

# 2. Definir el nombre del nuevo archivo
archivo_nuevo = FILE_FINAL_PARQUET.parent / "Resultados_Mapa_Emisiones_GPS_Kepler_Light.csv"

print(f"Escribiendo el nuevo archivo CSV de {len(df_ligero)} filas...")
df_ligero.to_csv(archivo_nuevo, index=False)

print(f"Archivo reducido guardado exitosamente en:\n{archivo_nuevo}")

Super Light

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Las columnas exactas (AGREGAMOS 'highway' PARA PODER FILTRAR LOS ERRORES)
columnas_mantener = [
    'caid', 'fecha_kepler', 'Speed [km/h]', 'highway',
    'geometry', 'distance_m', 'modo_transporte', 'Total_CO2_g'
]

print("Cargando el archivo original desde Parquet...")
# ALERTA DE OPTIMIZACIÓN: Leemos directo del Parquet solo las columnas necesarias
df_ligero = pd.read_parquet(FILE_FINAL_PARQUET, columns=columnas_mantener)

# ==============================================================================
# 2. FILTRO ESTÉTICO (OPCIÓN B: Ocultar saltos en el mapa)
# ==============================================================================
registros_antes = len(df_ligero)
# Nos quedamos solo con las calles reales, borrando las líneas rectas de error
df_ligero = df_ligero[df_ligero['highway'] != 'routing_error']
registros_despues = len(df_ligero)

print(f"Filtro estético aplicado: Se ocultaron {registros_antes - registros_despues:,} saltos de 'routing_error'.")

# ==============================================================================
# 3. GUARDADO FINAL
# ==============================================================================
# Usamos el .parent de tu variable global FILE_FINAL_PARQUET para guardarlo en la misma carpeta
archivo_nuevo = FILE_FINAL_PARQUET.parent / "Resultados_Mapa_Emisiones_GPS_Kepler_SuperLight.csv"

print(f"Escribiendo el nuevo archivo limpio de {len(df_ligero)} filas...")
df_ligero.to_csv(archivo_nuevo, index=False)

print(f"Archivo purgado y reducido exitosamente en:\n{archivo_nuevo}")